In [65]:
import pandas as pd
import os
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill
from openpyxl.utils.dataframe import dataframe_to_rows

# 기본 스타일 설정
default_font = Font(name='Arial', size=11)
default_alignment = Alignment(horizontal='general', vertical='bottom')
default_border = Border(left=Side(border_style='thin'), right=Side(border_style='thin'),
                        top=Side(border_style='thin'), bottom=Side(border_style='thin'))
default_fill = PatternFill(fill_type=None)

# 대상 폴더 경로 설정
folder_path = "N월별키워드검색추세5개메뉴단위"

# CSV 파일 읽기
csv_file = os.path.join(folder_path, '키워드_5개씩_정렬.csv')
df_csv = pd.read_csv(csv_file)

# 엑셀 파일 리스트 정렬
excel_files = sorted([f for f in os.listdir(folder_path) if f.startswith('datalab') and f.endswith('.xlsx')])

# 병합할 데이터 초기화
merged_df = pd.DataFrame()
first_dates = None

# 병합 루프
for idx, file in enumerate(excel_files):
    file_path = os.path.join(folder_path, file)
    
    # URL 추출 (B1 셀)
    url = pd.read_excel(file_path, nrows=1, header=None, usecols="B").iloc[0, 0]
    if idx < len(df_csv):
        df_csv.loc[idx, "URL"] = url
    
    # 메뉴 데이터 읽기
    df = pd.read_excel(file_path, header=6, nrows=26)
    
    # 첫 번째 파일의 날짜 열 저장
    if first_dates is None:
        first_dates = df.iloc[:, 0].copy()
    
    # 짝수 열만 추출 (실제 메뉴 데이터)
    even_cols = df.columns[1::2]
    df = df[even_cols]
    
    # 누적 병합
    merged_df = pd.concat([merged_df, df], axis=1)
    
    # CSV 파일에 URL 추가
    keyword_cols = df_csv.columns[1:]
    for col in keyword_cols:
        if col in df.columns:
            df_csv.loc[df_csv['그룹번호'] == idx+1, col + '_URL'] = url

# 최종 데이터프레임에 날짜 열 추가
merged_df = pd.concat([pd.DataFrame({"월": first_dates}), merged_df], axis=1)

# 날짜 열을 년월 형식으로 변환
merged_df['월'] = pd.to_datetime(merged_df['월']).dt.strftime('%Y-%m')

# 결과 저장
output_file = '병합된_상세메뉴_월별추세.xlsx'
wb = Workbook()

# 월별추세 시트 생성
ws1 = wb.active
ws1.title = "월별추세"
for r in dataframe_to_rows(merged_df, index=False, header=True):
    ws1.append(r)

# 키워드+URL 시트 생성
ws2 = wb.create_sheet("키워드+URL")
for r in dataframe_to_rows(df_csv, index=False, header=True):
    ws2.append(r)

# 스타일 적용
for ws in [ws1, ws2]:
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
        for cell in row:
            cell.font = default_font
            cell.alignment = default_alignment
            cell.border = default_border
            cell.fill = default_fill

wb.save(output_file)

print(f"\n병합 완료: '{output_file}' 파일 생성됨.")


병합 완료: '병합된_상세메뉴_월별추세.xlsx' 파일 생성됨.
